# U-Net

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

A refresher on the U-Net: an encoder–decoder convolutional network with **skip connections** that produces a per-pixel output the same size as its input. Originally for biomedical image segmentation (Ronneberger et al., 2015), now the workhorse backbone for segmentation and the denoiser inside most image diffusion models.

## 1. What & Why

**What it is.** U-Net is a fully-convolutional network shaped like a **U**: a contracting *encoder* that halves spatial resolution while doubling channels (capturing *what* is in the image), a *bottleneck*, then an expanding *decoder* that upsamples back to full resolution (recovering *where* things are). The defining trick is the **skip connections**: feature maps from each encoder level are concatenated onto the matching decoder level, so the decoder gets both deep semantic context *and* the high-resolution detail that pooling threw away.

**The problem it solves.** A plain classifier collapses an image to one label. Many tasks instead need a **dense prediction** — a label (or value) for *every pixel*: segment the tumor, the road, the cell nucleus; denoise an image; predict depth. Naively upsampling the bottleneck back to full size gives blurry, location-imprecise masks because spatial detail was destroyed by pooling. Skip connections fix exactly that.

**When to reach for it.**
- Image **segmentation** (semantic / binary / medical), especially with **few labeled examples** — U-Net was designed for small datasets and heavy augmentation.
- Any **image-to-image** map where output is spatially aligned with input: denoising, super-resolution, inpainting, depth/normal estimation, colorization.
- The **noise-prediction network in diffusion models** (Stable Diffusion's original backbone) — a U-Net augmented with attention and timestep conditioning.

**When not to.** Whole-image *classification* (use a plain CNN/ViT — you don't need the decoder), tasks needing global reasoning across distant regions where convolution's locality hurts (consider a Vision Transformer or a SegFormer/Mask2Former), or very high-resolution inputs where the full-res skip tensors blow up memory.

## 2. Mental Model

Picture an **hourglass with rungs across the gap**.

```
input ─►[enc1]──────────────skip──────────────►[dec1]─► output
          │ pool↓                            up↑ │
          └──►[enc2]────────skip────────►[dec2]──┘
                │ pool↓                up↑ │
                └──►[enc3]──skip──►[dec3]───┘
                      │ pool↓    up↑ │
                      └──►[bottleneck]┘
```

- Going **down** the left side you zoom out: lose resolution, gain abstraction — *"there is a cell here, somewhere."*
- The **bottleneck** holds the most compressed, most semantic view.
- Going **up** the right side you zoom back in, and at each level the **skip rung** hands the decoder the sharp edges from the encoder at the same scale — *"and the cell's boundary is exactly here."*

The decoder's job is to fuse *what* (from below) with *where* (from the skip) at every resolution. Without the rungs you get a soft, smeared mask; with them you get crisp boundaries.

## 3. Key Concepts

- **Encoder (contracting path).** Repeated `conv → conv → downsample`. Each step roughly halves H×W and doubles channels. Downsampling is usually `MaxPool2d(2)` or a stride-2 conv.
- **Bottleneck.** The lowest-resolution, highest-channel block at the bottom of the U. Largest receptive field.
- **Decoder (expanding path).** Repeated `upsample → concat skip → conv → conv`. Upsampling is a **transposed convolution** (`ConvTranspose2d`, learnable) or `Upsample` + a 1×1/3×3 conv (avoids checkerboard artifacts).
- **Skip connections.** Encoder feature maps **concatenated** (not added, unlike ResNet) onto the decoder at the matching resolution. This is the heart of U-Net.
- **Double conv block.** The standard unit: two 3×3 convs each followed by ReLU (modern variants insert BatchNorm/GroupNorm between conv and activation).
- **Same vs valid padding.** The original paper used *unpadded* (`valid`) convs, so output was smaller than input and skips were **center-cropped** to match. Modern implementations use `padding=1` (`same`) so sizes line up and no cropping is needed — the convention used here.
- **Output head.** A final 1×1 conv mapping to the desired number of channels: 1 logit for binary segmentation, `C` logits for `C` classes (then `softmax`/`argmax`), 3 for an RGB image.
- **Divisibility constraint.** With `k` pooling levels, input H and W must be divisible by `2**k` so upsampling reconstructs the exact original size.

## 4. Setup

Only PyTorch and NumPy are needed — everything runs on CPU in seconds. Uncomment the install if you're in a fresh environment.

In [1]:
# %pip install torch numpy
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| device: cpu")

torch 2.12.1 | device: cpu


## 5. Worked Examples

### Example 1 — Build a small U-Net and verify the shapes

We define a compact 3-level U-Net and confirm the two properties that matter: the **output is the same H×W as the input**, and each decoder block receives a **concatenated skip** (so its input channel count is encoder + upsampled).

In [2]:
def double_conv(in_ch, out_ch):
    """conv -> ReLU -> conv -> ReLU, the standard U-Net unit (same padding)."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=8):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        # encoder
        self.enc1 = double_conv(in_ch, base)
        self.enc2 = double_conv(base, base * 2)
        self.enc3 = double_conv(base * 2, base * 4)
        self.bottleneck = double_conv(base * 4, base * 8)
        # decoder: transposed conv upsamples, then concat skip, then double_conv
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = double_conv(base * 8, base * 4)   # base*4 (up) + base*4 (skip)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = double_conv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = double_conv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)        # 1x1 conv to output channels

    def forward(self, x):
        e1 = self.enc1(x)                 # full res
        e2 = self.enc2(self.pool(e1))     # 1/2
        e3 = self.enc3(self.pool(e2))     # 1/4
        b = self.bottleneck(self.pool(e3))  # 1/8
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))   # skip from e3
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # skip from e2
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # skip from e1
        return self.head(d1)


model = UNet(in_ch=1, out_ch=1, base=8)
x = torch.randn(2, 1, 64, 64)   # 64 is divisible by 2**3, the three pool levels
y = model(x)
n_params = sum(p.numel() for p in model.parameters())
print(f"input  {tuple(x.shape)}")
print(f"output {tuple(y.shape)}   <- same H,W as input")
print(f"parameters: {n_params:,}")

input  (2, 1, 64, 64)
output (2, 1, 64, 64)   <- same H,W as input
parameters: 120,681


### Example 2 — Train it to segment circles

A toy but honest segmentation task: synthetic 32×32 images, each a noisy background with one bright circle; the target is a binary mask of that circle. We train with `BCEWithLogitsLoss` (binary cross-entropy on raw logits, numerically stable) and watch the loss fall and pixel accuracy climb — proof the skip-connected decoder is recovering the circle's location and boundary.

In [3]:
def make_batch(n=8, size=32):
    """Random noisy images each with one bright disk; return (images, masks)."""
    imgs = np.random.rand(n, 1, size, size).astype("float32") * 0.3
    masks = np.zeros((n, 1, size, size), dtype="float32")
    yy, xx = np.ogrid[:size, :size]
    for i in range(n):
        cy, cx = np.random.randint(8, size - 8, size=2)
        r = np.random.randint(4, 8)
        disk = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
        imgs[i, 0][disk] += 0.7
        masks[i, 0][disk] = 1.0
    return torch.from_numpy(imgs), torch.from_numpy(masks)


torch.manual_seed(0)
np.random.seed(0)
model = UNet(in_ch=1, out_ch=1, base=8)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

for step in range(60):
    imgs, masks = make_batch(8, 32)
    opt.zero_grad()
    logits = model(imgs)
    loss = loss_fn(logits, masks)
    loss.backward()
    opt.step()
    if step % 15 == 0 or step == 59:
        acc = ((torch.sigmoid(logits) > 0.5) == (masks > 0.5)).float().mean()
        print(f"step {step:2d}  loss {loss.item():.4f}  pixel-acc {acc.item():.3f}")

step  0  loss 0.6360  pixel-acc 0.909
step 15  loss 0.5866  pixel-acc 0.908


step 30  loss 0.3711  pixel-acc 0.925
step 45  loss 0.2184  pixel-acc 0.892


step 59  loss 0.1256  pixel-acc 0.908


The loss drops and pixel accuracy rises toward ~1.0 in well under a second of CPU training — the network learns to copy the disk's position from the input into the output mask. Scale this up with real images, augmentation, and a deeper U (more levels / larger `base`) and it's the same architecture that wins medical-segmentation benchmarks.

## 6. Gotchas & Pitfalls

- **Input size must be divisible by `2**(#pool levels)`.** A 3-level U-Net needs H,W divisible by 8. Otherwise the upsampled tensor won't match its skip partner and `torch.cat` throws a size-mismatch error. Pad/resize inputs, or use `valid` convs with center-cropping (the original paper) — but `same` padding is far easier.
- **Concat vs add.** U-Net skips **concatenate** (channels grow); ResNet skips **add** (channels stay). Mixing these up gives wrong channel counts in the decoder's first conv — note `dec3` takes `base*8` = up(`base*4`) + skip(`base*4`).
- **Checkerboard artifacts** from `ConvTranspose2d` when kernel size isn't divisible by stride. Common fix: replace with `nn.Upsample(scale_factor=2)` + a 3×3 conv ("resize-convolution").
- **Class imbalance.** Segmentation masks are often mostly background. Plain BCE/cross-entropy lets the net predict "all background." Use **Dice loss**, focal loss, or `pos_weight` in `BCEWithLogitsLoss`.
- **Logits vs probabilities.** Use `BCEWithLogitsLoss` / `CrossEntropyLoss` on **raw logits**; don't apply your own `sigmoid`/`softmax` before the loss (double activation, unstable gradients). Apply `sigmoid`/`argmax` only at inference.
- **Forgetting normalization.** Deeper U-Nets train poorly without BatchNorm/GroupNorm in the conv blocks. GroupNorm is preferred when batch size is tiny (common in 3D/medical).
- **Memory.** Full-resolution skip tensors are large; high-res inputs blow up activation memory. Tile/patch the image, use mixed precision, or downsample more aggressively.
- **Wrong final channel count.** Binary segmentation = 1 logit + BCE; multi-class = `C` logits + cross-entropy. Don't use `C=2` with BCE.

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|
| **U-Net** | Crisp dense output, data-efficient, simple, strong with augmentation | Convolution is local; struggles with very large objects / long-range context | Medical & general segmentation, image-to-image, diffusion backbone |
| **FCN / DeepLab (ASPP)** | Atrous convolutions enlarge receptive field without losing resolution | Heavier, less detail recovery than skip-rich U-Net | Natural-scene semantic segmentation at scale |
| **Vision Transformer / SegFormer / Mask2Former** | Global attention, SOTA on big datasets | Data- and compute-hungry; weaker on small datasets | Large labeled datasets, need long-range reasoning |
| **Plain CNN / ResNet classifier** | Simple, cheap | One label per image, no spatial output | Whole-image classification, not dense prediction |
| **U-Net++ / Attention U-Net / 3D U-Net** | Denser skips, gated skips, volumetric data | More params/compute | Squeeze out accuracy; 3D volumes (CT/MRI) |

**Rule of thumb:** if you need a per-pixel output spatially aligned with the input and don't have millions of labels, start with U-Net. Move to transformer-based segmenters only when you have the data/compute and need global context. For 3D medical volumes, use **nnU-Net** — a self-configuring U-Net pipeline that is still a top baseline.

## 8. Resources

- **Original paper** — Ronneberger, Fischer, Brox, *U-Net: Convolutional Networks for Biomedical Image Segmentation* (2015): https://arxiv.org/abs/1505.04597
- **PyTorch implementation (milesial/Pytorch-UNet)** — clean, trainable reference: https://github.com/milesial/Pytorch-UNet
- **nnU-Net** — self-configuring U-Net framework, still a SOTA segmentation baseline: https://github.com/MIC-DKFZ/nnUNet
- **Distill, *Deconvolution and Checkerboard Artifacts*** — why transposed conv artifacts happen and how to avoid them: https://distill.pub/2016/deconv-checkerboard/
- **U-Net++** — Zhou et al., *A Nested U-Net Architecture* (2018): https://arxiv.org/abs/1807.10165
- **Diffusion U-Net** — Ho et al., *Denoising Diffusion Probabilistic Models* (2020), where U-Net is the noise predictor: https://arxiv.org/abs/2006.11239